In [ ]:
# Import Tensorflow 2.0
#%tensorflow_version 2.x
import tensorflow as tf

# Download and import the MIT 6.S191 package
!pip install mitdeeplearning
import mitdeeplearning as mdl

# Import all remaining packages
import numpy as np
import os
import time
import functools
from IPython import display as ipythondisplay
from tqdm import tqdm
!apt-get install abcmidi timidity > /dev/null 2>&1

## 1.2 Dataset

In [ ]:
# Download the dataset
songs = mdl.lab1.load_training_data()

# Print one of the songs to inspect it in greater detail!
example_song = songs[0]
print("\nExample song: ")
print(example_song)

In [ ]:
# Convert the ABC notation to audio file and listen to it
mdl.lab1.play_song(example_song)

In [ ]:
# Join our list of song strings into a single string containing all songs
songs_joined = "\n\n".join(songs)

# Find all unique characters in the joined string
vocab = sorted(set(songs_joined))
print("There are", len(vocab), "unique characters in the dataset")

## 1.3 Process the dataset for the learning task

### Vectorize the text

In [ ]:
### Define numerical representation of text ###

# Create a mapping from character to unique index.
# For example, to get the index of the character "d",
#   we can evaluate `char2idx["d"]`.
char2idx = {u:i for i, u in enumerate(vocab)}

# Create a mapping from indices to characters. This is
#   the inverse of char2idx and allows us to convert back
#   from unique index to the character in our vocabulary.
idx2char = np.array(vocab)

In [ ]:
print('{')
for char,_ in zip(char2idx, range(20)):
    print('  {:4s}: {:3d},'.format(repr(char), char2idx[char]))
print('  ...\n}')

In [ ]:
### Vectorize the songs string ###

'''TODO: Write a function to convert the all songs string to a vectorized
    (i.e., numeric) representation. Use the appropriate mapping
    above to convert from vocab characters to the corresponding indices.

  NOTE: the output of the `vectorize_string` function
  should be a np.array with `N` elements, where `N` is
  the number of characters in the input string
'''

def vectorize_string(string):
  vectorized_songs = np.array([char2idx[song] for song in string ])
  return vectorized_songs
vectorized_songs = vectorize_string(songs_joined)

In [ ]:
print ('{} ---- characters mapped to int ----> {}'.format(repr(songs_joined[:10]), vectorized_songs[:10]))
# check that vectorized_songs is a numpy array
assert isinstance(vectorized_songs, np.ndarray), "returned result should be a numpy array"

### Create training examples and targets

In [ ]:
### Batch definition to create training examples ###

def get_batch(vectorized_songs, seq_length, batch_size):
  # the length of the vectorized songs string
  n = vectorized_songs.shape[0] - 1
  # randomly choose the starting indices for the examples in the training batch
  idx = np.random.choice(n-seq_length, batch_size)

  '''TODO: construct a list of input sequences for the training batch'''
  input_batch = [vectorized_songs[i:i+seq_length] for i in idx]
  '''TODO: construct a list of output sequences for the training batch'''
  output_batch = [vectorized_songs[i+1:i+seq_length+1] for i in idx]

  # x_batch, y_batch provide the true inputs and targets for network training
  x_batch = np.reshape(input_batch, [batch_size, seq_length])
  y_batch = np.reshape(output_batch, [batch_size, seq_length])
  return x_batch, y_batch


# Perform some simple tests to make sure your batch function is working properly!
test_args = (vectorized_songs, 10, 2)
if not mdl.lab1.test_batch_func_types(get_batch, test_args) or \
   not mdl.lab1.test_batch_func_shapes(get_batch, test_args) or \
   not mdl.lab1.test_batch_func_next_step(get_batch, test_args):
   print("======\n[FAIL] could not pass tests")
else:
   print("======\n[PASS] passed all tests!")

In [ ]:
x_batch, y_batch = get_batch(vectorized_songs, seq_length=5, batch_size=1)

for i, (input_idx, target_idx) in enumerate(zip(np.squeeze(x_batch), np.squeeze(y_batch))):
    print("Step {:3d}".format(i))
    print("  input: {} ({:s})".format(input_idx, repr(idx2char[input_idx])))
    print("  expected output: {} ({:s})".format(target_idx, repr(idx2char[target_idx])))

## 1.4 Define the Long Short-Term Memory (LSTM) model

In [ ]:
def build_model(vocab_size, embedding_dim, rnn_units, batch_size):
    return tf.keras.Sequential([
        tf.keras.layers.Embedding(input_dim=vocab_size, output_dim=embedding_dim),
        tf.keras.layers.LSTM(
            rnn_units,
            return_sequences=True,
            recurrent_initializer='glorot_uniform',
            recurrent_activation='sigmoid',
            stateful=True  # Keep state across batches for music continuity
        ),
        tf.keras.layers.Dense(vocab_size)
    ])

# Loss function
def loss(labels, logits):
    return tf.keras.losses.sparse_categorical_crossentropy(labels, logits, from_logits=True)

# Example model build and compile
vocab_size = 10000
embedding_dim = 256
rnn_units = 1024
batch_size = 64

model = build_model(vocab_size, embedding_dim, rnn_units, batch_size)
model.compile(optimizer='adam', loss=loss)
model.summary()

### Test out the LSTM model

In [ ]:
model.summary()

In [ ]:
x, y = get_batch(vectorized_songs, seq_length=100, batch_size=32)
pred = model(x)
print("Input shape:      ", x.shape, " # (batch_size, sequence_length)")
print("Prediction shape: ", pred.shape, "# (batch_size, sequence_length, vocab_size)")

### Predictions from the untrained model

In [ ]:
sampled_indices = tf.random.categorical(pred[0], num_samples=1)
sampled_indices = tf.squeeze(sampled_indices,axis=-1).numpy()
sampled_indices

In [ ]:
x, y = get_batch(vectorized_songs, seq_length=100, batch_size=32)
pred = model(x)
print("Input shape:      ", x.shape, " # (batch_size, sequence_length)")
print("Prediction shape: ", pred.shape, "# (batch_size, sequence_length, vocab_size)")

## 1.5 Training the model: loss and training operations

In [ ]:
### Defining the loss function ###

'''TODO: define the loss function to compute and return the loss between
    the true labels and predictions (logits). Set the argument from_logits=True.'''
def compute_loss(labels, logits):
  loss = tf.keras.losses.sparse_categorical_crossentropy(labels, logits, from_logits=True) # TODO
  return loss

'''TODO: compute the loss using the true next characters from the example batch
    and the predictions from the untrained model several cells above'''
example_batch_loss = compute_loss(y, pred) # TODO

print("Prediction shape: ", pred.shape, " # (batch_size, sequence_length, vocab_size)")
print("scalar_loss:      ", example_batch_loss.numpy().mean())

In [ ]:
### Hyperparameter setting and optimization ###

# Optimization parameters:
num_training_iterations = 50000  # Increase this to train longer
batch_size = 64  # Experiment between 1 and 64
seq_length = 100  # Experiment between 50 and 500
learning_rate = 1e-3  # Experiment between 1e-5 and 1e-1

# Model parameters:
vocab_size = len(vocab)
embedding_dim = 256
rnn_units = 1024  # Experiment between 1 and 2048

# Checkpoint location:
checkpoint_dir = './training_checkpoints'
checkpoint_prefix = os.path.join(checkpoint_dir, "my_ckpt")

In [ ]:
### Define optimizer and training operation ###

'''TODO: instantiate a new model for training using the `build_model`
  function and the hyperparameters created above.'''
model = build_model(vocab_size, embedding_dim, rnn_units, batch_size)

'''TODO: instantiate an optimizer with its learning rate.
  Checkout the tensorflow website for a list of supported optimizers.
  https://www.tensorflow.org/api_docs/python/tf/keras/optimizers/
  Try using the Adam optimizer to start.'''
optimizer = tf.keras.optimizers.Adam(learning_rate)

@tf.function
def train_step(x, y):
  # Use tf.GradientTape()
  with tf.GradientTape() as tape:

    '''TODO: feed the current input into the model and generate predictions'''
    y_hat = model(x)

    '''TODO: compute the loss!'''
    loss = compute_loss(y, y_hat)

  # Now, compute the gradients
  '''TODO: complete the function call for gradient computation.
      Remember that we want the gradient of the loss with respect all
      of the model parameters.
      HINT: use `model.trainable_variables` to get a list of all model
      parameters.'''
  grads = tape.gradient(loss, model.trainable_variables)

  # Apply the gradients to the optimizer so it can update the model accordingly
  optimizer.apply_gradients(zip(grads, model.trainable_variables))
  return loss

##################
# Begin training!#
##################

history = []
plotter = mdl.util.PeriodicPlotter(sec=2, xlabel='Iterations', ylabel='Loss')
if hasattr(tqdm, '_instances'): tqdm._instances.clear() # clear if it exists

# Create the checkpoint directory if it doesn't exist
import os
os.makedirs(checkpoint_dir, exist_ok=True)  # Create the directory if it doesn't exist

for iter in tqdm(range(num_training_iterations)):

  # Grab a batch and propagate it through the network
  x_batch, y_batch = get_batch(vectorized_songs, seq_length, batch_size)
  loss = train_step(x_batch, y_batch)

  # Update the progress bar
  history.append(loss.numpy().mean())
  plotter.plot(history)
 # Update the model with the changed weights!
  if iter % 100 == 0:
    model.save_weights(checkpoint_prefix + '.weights.h5') # Add the '.weights.h5' extension to the filename

# Save the trained model and the weights
model.save_weights(checkpoint_prefix + '.weights.h5') # Add the '.weights.h5' extension to the filename

## 1.6 Evaluation of Model

In [ ]:
import tensorflow as tf
import numpy as np

def calculate_accuracy(y_true, y_pred):
    """Calculate the accuracy of predictions."""
    y_pred = tf.argmax(y_pred, axis=-1)  # Shape: (batch_size, seq_length)
    correct_predictions = tf.equal(y_true, y_pred)
    accuracy = tf.reduce_mean(tf.cast(correct_predictions, tf.float32))
    return accuracy

def evaluate_accuracy(model, dataset):
    """Evaluate the accuracy of the model on the given dataset."""
    accuracies = []

    for x_batch, y_batch in dataset:
        y_pred = model(x_batch, training=False)
        accuracy = calculate_accuracy(y_batch, y_pred)
        accuracies.append(accuracy.numpy())  # Move numpy() here

    mean_accuracy = np.mean(accuracies)
    print(f"Accuracy: {mean_accuracy:.4f}")
    return mean_accuracy

# Generate a dataset for evaluation
x_val, y_val = get_batch(vectorized_songs, seq_length, batch_size)
val_dataset = [(x_val, y_val)]  # or use tf.data.Dataset for multiple batches

# Evaluate the model's accuracy
mean_accuracy = evaluate_accuracy(model, val_dataset)

In [ ]:
import tensorflow as tf
import numpy as np

# --- Note-level accuracy ---
def compute_accuracy(labels, logits):
    labels = tf.cast(labels, tf.int64)
    predictions = tf.argmax(logits, axis=2)
    accuracy = tf.equal(predictions, labels)
    accuracy = tf.cast(accuracy, tf.float32)
    return tf.reduce_mean(accuracy).numpy()

# --- Cross-entropy loss ---
def compute_loss(y_true, y_pred):
    loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True, reduction='none')
    return loss_fn(y_true, y_pred)

# --- Perplexity ---
def calculate_perplexity(y_true, y_pred):
    loss = compute_loss(y_true, y_pred)
    return tf.exp(tf.reduce_mean(loss)).numpy()

# --- Dataset creation from full vectorized data ---
def create_dataset(data, seq_length=100, batch_size=32):
    n = len(data) - 1
    inputs = [data[i:i+seq_length] for i in range(0, n - seq_length, seq_length)]
    targets = [data[i+1:i+seq_length+1] for i in range(0, n - seq_length, seq_length)]
    x = np.array(inputs)
    y = np.array(targets)
    return tf.data.Dataset.from_tensor_slices((x, y)).batch(batch_size, drop_remainder=True)

# --- Evaluation function ---
def evaluate_model(model, dataset):
    total_loss = 0.0
    total_note_accuracy = 0.0
    total_perplexity = 0.0
    num_batches = 0

    for x_batch, y_batch in dataset:
        y_pred = model(x_batch, training=False)

        loss = compute_loss(y_batch, y_pred)
        note_acc = compute_accuracy(y_batch, y_pred)
        perplex = calculate_perplexity(y_batch, y_pred)

        total_loss += tf.reduce_mean(loss).numpy()
        total_note_accuracy += note_acc
        total_perplexity += perplex
        num_batches += 1

    avg_loss = total_loss / num_batches
    avg_note_accuracy = total_note_accuracy / num_batches
    avg_perplexity = total_perplexity / num_batches

    print(f"\n Evaluation Results:")
    print(f"- Average Loss     : {avg_loss:.4f}")
    print(f"- Note Accuracy    : {avg_note_accuracy:.4f}")
    print(f"- Perplexity       : {avg_perplexity:.4f}")

    return avg_note_accuracy, avg_perplexity

# --- Run Evaluation ---
full_dataset = create_dataset(vectorized_songs, seq_length=100, batch_size=64)
avg_note_acc, avg_perplex = evaluate_model(model, full_dataset)

## 1.7 Generate music using the LSTM model

In [ ]:
'''TODO: Rebuild the model using a batch_size=1'''
model = build_model(vocab_size, embedding_dim, rnn_units, batch_size=1)

# Provide the correct path to the checkpoint directory
checkpoint_dir = './training_checkpoints'  # Update with the actual directory path

# Verify the directory exists and contains checkpoint files
!ls {checkpoint_dir}

# Restore the model weights for the last checkpoint after training
latest_checkpoint = tf.train.latest_checkpoint(checkpoint_dir)
if latest_checkpoint:
    model.load_weights(latest_checkpoint)
    model.build(tf.TensorShape([1, None]))
    model.summary()
else:
    print(f"Error: No checkpoints found in {checkpoint_dir}")

In [ ]:
import os
import random
import numpy as np
import tensorflow as tf
from IPython.display import Audio, display
from tqdm import tqdm
import re

def fix_abc_rhythm(abc_string):
    """
    Fixes ABC rhythm to ensure:
    - No unnatural gaps
    - Smooth, continuous, natural tempo
    - Fixed or alternating durations
    """
    import re

    # Extract header (or use default)
    header_match = re.search(r'(X:.*?K:[^\n]+\n)', abc_string, re.DOTALL)
    header = header_match.group(1) if header_match else "X:1\nT:Generated\nM:4/4\nL:1/8\nK:C\n"

    # Add tempo for smoother playback
    if "Q:" not in header:
        header += "Q:1/4=30\n"  # Adjust this tempo for slower/faster pace

    body = abc_string.replace(header, "") if header else abc_string

    # Remove unwanted characters
    body = re.sub(r'[zZxX]|[|:\[\]]+', '', body)

    # Extract notes
    notes = re.findall(r"[A-Ga-g][,']*", body)

    # Alternate durations for musicality
    durations = ["1/4", "1/8", "1/4", "1/8"]
    cleaned_notes = [note + durations[i % len(durations)] for i, note in enumerate(notes)]

    # Group into measures
    measures = [" ".join(cleaned_notes[i:i+8]) for i in range(0, len(cleaned_notes), 8)]
    final_body = " |\n".join(measures) + " |"

    return header + final_body + "\n"


# Generate a song using trained model
def generate_song_from_model(model, char2idx, idx2char, start_string, generation_length=2000, temperature=0.5):
    input_eval = [char2idx.get(c, 0) for c in start_string]
    input_eval = tf.expand_dims(input_eval, 0)

    generated_song = []
    if hasattr(model, 'reset_states'):
        model.reset_states()

    for _ in tqdm(range(generation_length), desc="Generating"):
        predictions = model(input_eval)
        predictions = tf.squeeze(predictions, 0)
        predictions = predictions / temperature
        predicted_id = tf.random.categorical(predictions, num_samples=1)[-1, 0].numpy()
        generated_song.append(idx2char[predicted_id])
        input_eval = tf.expand_dims([predicted_id], 0)

    raw_output = start_string + ''.join(generated_song)
    cleaned_output = fix_abc_rhythm(raw_output)
    return cleaned_output

# Convert ABC to audio
def convert_abc_to_audio(abc_text, filename):
    try:
        abc_path = f"{filename}.abc"
        with open(abc_path, "w") as f:
            f.write(abc_text)

        midi_path = f"{filename}.mid"
        os.system(f"abc2midi {abc_path} -o {midi_path} > /dev/null 2>&1")
        print(f" MIDI saved: {midi_path}")

        wav_path = f"{filename}.wav"
        os.system(f"timidity {midi_path} -Ow -o {wav_path} > /dev/null 2>&1")
        print(f" WAV saved: {wav_path}")

        return wav_path
    except Exception as e:
        print(f"Error converting {filename}: {e}")
        return None

# Main function
def main():
    print("Loading dataset and model...")
    import mitdeeplearning as mdl
    songs = mdl.lab1.load_training_data()

    vocab = sorted(set("".join(songs)))
    char2idx = {u: i for i, u in enumerate(vocab)}
    idx2char = np.array(vocab)

    # Define the model
    class LSTMModel(tf.keras.Model):
        def __init__(self, vocab_size, embedding_dim=256, rnn_units=1024):
            super().__init__()
            self.embedding = tf.keras.layers.Embedding(vocab_size, embedding_dim)
            self.lstm = tf.keras.layers.LSTM(rnn_units, return_sequences=True, stateful=True)
            self.dense = tf.keras.layers.Dense(vocab_size)

        def call(self, inputs):
            x = self.embedding(inputs)
            x = self.lstm(x)
            return self.dense(x)

    # Load model
    model = LSTMModel(len(vocab))
    try:
        model.build((1, None))
        model.load_weights("./training_checkpoints/my_ckpt.weights.h5")
        print(" Model loaded successfully")
    except Exception as e:
        print(f" Failed to load model: {e}")
        return

    # Pick original song
    original_song = random.choice(songs)
    print("\n Original song:")
    print(original_song)

    seed_string = original_song[:50]
    print("\n Seed string:")
    print(seed_string)

    # Generate new song
    print("\n Generating new song...")
    generated_song = generate_song_from_model(
        model, char2idx, idx2char,
        start_string=seed_string,
        generation_length=1500,
        temperature=0.5
    )

    print("\n Generated song preview:")
    print(generated_song[:300] + "...")

    # Save generated song to file
    with open("generated_song_cleaned.abc", "w") as f:
        f.write(generated_song)

    # Convert to audio
    print("\n Converting original song to audio...")
    original_audio_path = convert_abc_to_audio(original_song, "original_song")

    print("\n Converting generated song to audio...")
    generated_audio_path = convert_abc_to_audio(generated_song, "generated_song")

    # Play audio
    if original_audio_path:
        print("\n Playing original song:")
        display(Audio(filename=original_audio_path))

    if generated_audio_path:
        print("\n Playing generated song:")
        display(Audio(filename=generated_audio_path))

# Run it
if __name__ == "__main__":
    main()

#Trying to generate same song

In [ ]:
import os
import random
import numpy as np
import tensorflow as tf
from IPython.display import Audio, display
from tqdm import tqdm
import re

# Fix ABC rhythm
def fix_abc_rhythm(abc_string):
    header_match = re.search(r'(X:.*?K:[^\n]+\n)', abc_string, re.DOTALL)
    header = header_match.group(1) if header_match else "X:1\nT:Generated\nM:4/4\nL:1/8\nK:C\n"

    if "Q:" not in header:
        header += "Q:1/4=30\n"

    body = abc_string.replace(header, "") if header else abc_string
    body = re.sub(r'[zZxX]|[|:\[\]]+', '', body)
    notes = re.findall(r"[A-Ga-g][,']*", body)

    durations = ["1/4", "1/8", "1/4", "1/8"]
    cleaned_notes = [note + durations[i % len(durations)] for i, note in enumerate(notes)]

    measures = [" ".join(cleaned_notes[i:i+8]) for i in range(0, len(cleaned_notes), 8)]
    final_body = " |\n".join(measures) + " |"

    return header + final_body + "\n"

# Generate song from model
def generate_song_from_model(model, char2idx, idx2char, start_string, generation_length=0, temperature=0.0):
    input_eval = [char2idx.get(c, 0) for c in start_string]
    input_eval = tf.expand_dims(input_eval, 0)

    generated_song = []
    if hasattr(model, 'reset_states'):
        model.reset_states()

    for _ in tqdm(range(generation_length), desc="Generating"):
        predictions = model(input_eval)
        predictions = tf.squeeze(predictions, 0)
        predictions = predictions / temperature
        predicted_id = tf.random.categorical(predictions, num_samples=1)[-1, 0].numpy()
        generated_song.append(idx2char[predicted_id])
        input_eval = tf.expand_dims([predicted_id], 0)

    raw_output = start_string + ''.join(generated_song)
    cleaned_output = fix_abc_rhythm(raw_output)
    return cleaned_output

# Convert ABC to audio
def convert_abc_to_audio(abc_text, filename):
    try:
        abc_path = f"{filename}.abc"
        with open(abc_path, "w") as f:
            f.write(abc_text)

        midi_path = f"{filename}.mid"
        os.system(f"abc2midi {abc_path} -o {midi_path} > /dev/null 2>&1")
        print(f" MIDI saved: {midi_path}")

        wav_path = f"{filename}.wav"
        os.system(f"timidity {midi_path} -Ow -o {wav_path} > /dev/null 2>&1")
        print(f" WAV saved: {wav_path}")

        return wav_path
    except Exception as e:
        print(f" Error converting {filename}: {e}")
        return None

# Main function
def main():
    print(" Loading dataset and model...")
    import mitdeeplearning as mdl
    songs = mdl.lab1.load_training_data()

    vocab = sorted(set("".join(songs)))
    char2idx = {u: i for i, u in enumerate(vocab)}
    idx2char = np.array(vocab)

    # Define the model
    class LSTMModel(tf.keras.Model):
        def __init__(self, vocab_size, embedding_dim=256, rnn_units=1024):
            super().__init__()
            self.embedding = tf.keras.layers.Embedding(vocab_size, embedding_dim)
            self.lstm = tf.keras.layers.LSTM(rnn_units, return_sequences=True, stateful=True)
            self.dense = tf.keras.layers.Dense(vocab_size)

        def call(self, inputs):
            x = self.embedding(inputs)
            x = self.lstm(x)
            return self.dense(x)

    # Load model
    model = LSTMModel(len(vocab))
    try:
        model.build((1, None))
        model.load_weights("./training_checkpoints/my_ckpt.weights.h5")
        print(" Model loaded successfully")
    except Exception as e:
        print(f" Failed to load model: {e}")
        return

    # Use song from dataset
    original_song = random.choice(songs)
    print("\n Original song:")
    print(original_song)

    seed_string = original_song
    print("\n Seed string (entire song):")
    print(seed_string)

    # Generate song to match original
    print("\n Generating song to match original...")
    generated_song = generate_song_from_model(
        model, char2idx, idx2char,
        start_string=seed_string,
        generation_length=0,
        temperature=0.0
    )

    print("\n Generated song preview:")
    print(generated_song[:300] + "...")

    # Save generated song
    with open("generated_song_cleaned.abc", "w") as f:
        f.write(generated_song)

    # Convert to audio
    print("\n Converting original song to audio...")
    original_audio_path = convert_abc_to_audio(original_song, "original_song")

    print("\n Converting generated song to audio...")
    generated_audio_path = convert_abc_to_audio(generated_song, "generated_song")

    # Play audio
    if original_audio_path:
        print("\n Playing original song:")
        display(Audio(filename=original_audio_path))

    if generated_audio_path:
        print("\n Playing generated song:")
        display(Audio(filename=generated_audio_path))

# Run
if __name__ == "__main__":
    main()

## 1.8 Evaluation: Comparing Generated and Original Songs
### Metrics: Jaccard, Pitch Histogram, Repetition, n-gram, Contour,Coherence, BLEU Score, Sequence Similarity


In [ ]:
from music21 import converter, note
from collections import Counter
from difflib import SequenceMatcher
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
import numpy as np

# --- Load ABC from file ---
def load_abc_from_file(filepath):
    try:
        with open(filepath, 'r') as f:
            return f.read()
    except Exception as e:
        print(f" Error reading {filepath}: {e}")
        return ""

# --- Parse ABC into note list ---
def parse_abc_notes(abc_text):
    try:
        parsed = converter.parse(abc_text, format='abc')
        notes = parsed.flatten().notes
        return [n.nameWithOctave for n in notes if isinstance(n, note.Note)]
    except Exception as e:
        print(f" Error parsing ABC: {e}")
        return []

# --- Jaccard Note Overlap ---
def jaccard_overlap(a, b):
    set_a, set_b = set(a), set(b)
    return len(set_a & set_b) / len(set_a | set_b) if (set_a | set_b) else 0

# --- Pitch Class Histogram Δ ---
def pitch_class_histogram(notes):
    h = {p: 0 for p in "CDEFGAB"}
    for n in notes:
        if n[0] in h:
            h[n[0]] += 1
    total = sum(h.values()) or 1
    return {k: v / total for k, v in h.items()}

def histogram_difference(h1, h2):
    return sum(abs(h1[k] - h2[k]) for k in h1)

# --- Repetition Factor ---
def repetition_factor(notes):
    return sum(notes.count(n) for n in set(notes)) / len(notes) if notes else 0

# --- n-gram Overlap ---
def ngram_overlap(a, b, n=2):
    def extract_ngrams(seq, n):
        return Counter(tuple(seq[i:i+n]) for i in range(len(seq) - n + 1))
    a_ngrams = extract_ngrams(a, n)
    b_ngrams = extract_ngrams(b, n)
    shared = sum((a_ngrams & b_ngrams).values())
    total = sum((a_ngrams | b_ngrams).values())
    return shared / total if total else 0

# --- Melodic Contour Similarity ---
def melodic_contour(notes):
    def to_midi(n):
        try: return note.Note(n).pitch.midi
        except: return None
    mids = [to_midi(n) for n in notes if n]
    mids = [m for m in mids if m is not None]
    contour = []
    for i in range(1, len(mids)):
        if mids[i] > mids[i-1]: contour.append('U')
        elif mids[i] < mids[i-1]: contour.append('D')
        else: contour.append('S')
    return contour

def contour_similarity(c1, c2):
    return SequenceMatcher(None, c1, c2).ratio()

# --- Musical Coherence ---
def musical_coherence(notes):
    def to_midi(n):
        try: return note.Note(n).pitch.midi
        except: return None
    mids = [to_midi(n) for n in notes if n]
    mids = [m for m in mids if m is not None]
    if len(mids) < 2: return 1.0
    intervals = [mids[i+1] - mids[i] for i in range(len(mids) - 1)]
    std_dev = np.std(intervals)
    max_possible = 87  # MIDI 21–108 range
    return 1.0 - min(1.0, std_dev / max_possible)

# --- BLEU Score ---
def compute_bleu(reference, candidate):
    try:
        smoothing = SmoothingFunction().method1
        return sentence_bleu([reference], candidate, smoothing_function=smoothing)
    except Exception:
        return 0.0

# --- Sequence Similarity (Levenshtein-like) ---
def sequence_similarity(a, b):
    return SequenceMatcher(None, a, b).ratio()

# --- Evaluate ABC-Based Generation ---
def evaluate_similarity(original_notes, generated_notes):
    if not original_notes or not generated_notes:
        print(" One or both note sequences are empty. Check ABC content.")
        return

    print("\n Evaluation Summary:")
    print(f" Note Overlap (Jaccard):        {jaccard_overlap(original_notes, generated_notes):.2f}")

    h_orig = pitch_class_histogram(original_notes)
    h_gen = pitch_class_histogram(generated_notes)
    print(f" Pitch Class Histogram Δ:       {histogram_difference(h_orig, h_gen):.2f} (lower is better)")

    print(f" Repetition Factor (original):  {repetition_factor(original_notes):.2f}")
    print(f" Repetition Factor (generated): {repetition_factor(generated_notes):.2f}")

    print(f" Bigram Overlap:                {ngram_overlap(original_notes, generated_notes, 2):.2f}")
    print(f" Trigram Overlap:               {ngram_overlap(original_notes, generated_notes, 3):.2f}")

    contour_orig = melodic_contour(original_notes)
    contour_gen = melodic_contour(generated_notes)
    print(f" Melodic Contour Similarity:    {contour_similarity(contour_orig, contour_gen):.2f}")

    print(f" Musical Coherence (original):  {musical_coherence(original_notes):.2f}")
    print(f" Musical Coherence (generated): {musical_coherence(generated_notes):.2f}")

    print(f" BLEU Score (ABC Tokens):       {compute_bleu(original_notes, generated_notes):.2f}")
    print(f" Sequence Similarity:           {sequence_similarity(original_notes, generated_notes):.2f}")

# --- Load and evaluate ---
original_song = load_abc_from_file("/content/original_song.abc")
generated_song = load_abc_from_file("/content/generated_song_cleaned.abc")

original_notes = parse_abc_notes(original_song)
generated_notes = parse_abc_notes(generated_song)

evaluate_similarity(original_notes, generated_notes)

#Confusion Matrix

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
from music21 import converter, note
from collections import Counter # Keep this import as it's used in the original cell

# --- Load ABC from file ---
def load_abc_from_file(filepath):
    try:
        with open(filepath, 'r') as f:
            return f.read()
    except Exception as e:
        print(f" Error reading {filepath}: {e}")
        return ""

# --- Parse ABC into note list ---
def parse_abc_notes(abc_text):
    try:
        parsed = converter.parse(abc_text, format='abc')
        notes = parsed.flat.notes
        return [n.nameWithOctave for n in notes if isinstance(n, note.Note)]
    except Exception as e:
        print(f" Error parsing ABC: {e}")
        return []

# --- Convert note sequence to IDs and labels ---
def note_sequence_to_ids(seq):
    unique = sorted(set(seq))
    note_to_id = {n: i for i, n in enumerate(unique)}
    return [note_to_id[n] for n in seq], list(note_to_id.keys())

# --- Confusion Matrix Plot ---
def plot_confusion_matrix(y_true_notes, y_pred_notes, title="Confusion Matrix"):
    y_true_ids, labels = note_sequence_to_ids(y_true_notes)
    y_pred_ids = [labels.index(n) if n in labels else -1 for n in y_pred_notes]

    # Filter out unmatched predictions
    y_true_filtered, y_pred_filtered = [], []
    for yt, yp in zip(y_true_ids, y_pred_ids):
        if yp != -1:
            y_true_filtered.append(yt)
            y_pred_filtered.append(yp)

    cm = confusion_matrix(y_true_filtered, y_pred_filtered, labels=range(len(labels)))
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels)
    plt.xlabel("Predicted Notes")
    plt.ylabel("True Notes")
    plt.title(title)
    plt.tight_layout()
    plt.show()

# --- Example Usage ---
if __name__ == "__main__":
    # from abc_evaluation_script import load_abc_from_file, parse_abc_notes # Removed the problematic import

    original_abc = load_abc_from_file("/content/original_song.abc")
    generated_abc = load_abc_from_file("/content/generated_song_cleaned.abc")

    original_notes = parse_abc_notes(original_abc)
    generated_notes = parse_abc_notes(generated_abc)

    plot_confusion_matrix(original_notes, generated_notes)

#Audio Similarity(DTW)

In [ ]:
# audio_similarity_eval.py
import librosa
import numpy as np
from librosa.sequence import dtw

# --- Audio Similarity using DTW over MFCCs ---
def compute_audio_similarity_dtw(file1, file2, sr=22050, n_mfcc=20):
    try:
        # Load audio
        y1, _ = librosa.load(file1, sr=sr)
        y2, _ = librosa.load(file2, sr=sr)

        # Extract MFCCs (shape: n_mfcc × time)
        mfcc1 = librosa.feature.mfcc(y=y1, sr=sr, n_mfcc=n_mfcc)
        mfcc2 = librosa.feature.mfcc(y=y2, sr=sr, n_mfcc=n_mfcc)

        # Compute DTW (match MFCC features over time using cosine distance)
        D, wp = dtw(X=mfcc1, Y=mfcc2, metric='cosine')

        # Normalize distance by length of path
        dist = D[-1, -1] / len(wp)
        similarity = 1 / (1 + dist)  # Higher is better
        print(f"DTW Audio Similarity: {similarity:.2f}")
        return similarity

    except Exception as e:
        print(f"Error during DTW audio comparison: {e}")
        return 0.0

# --- Example Usage ---
if __name__ == "__main__":
    file1 = "/content/original_song.wav"
    file2 = "/content/generated_song.wav"
    compute_audio_similarity_dtw(file1, file2)

#Music Spectogram

In [ ]:
import librosa
import librosa.display
import matplotlib.pyplot as plt
import numpy as np

# Load the original audio file
original_audio_path = "/content/original_song.wav"  # Or "original_song.mp3"
y_orig, sr_orig = librosa.load(original_audio_path, sr=None)

# Create and display spectrogram for original music
plt.figure(figsize=(10, 4))
S_orig = librosa.feature.melspectrogram(y=y_orig, sr=sr_orig)
librosa.display.specshow(librosa.power_to_db(S_orig, ref=np.max),
                         sr=sr_orig, x_axis='time', y_axis='mel')
plt.title("Original Music Spectrogram")
plt.colorbar(format="%+2.0f dB")
plt.tight_layout()
plt.show()

# Load generated audio file
generated_audio_path = "/content/generated_song.wav"  # or .wav
y_gen, sr_gen = librosa.load(generated_audio_path, sr=None)

# Create and display spectrogram for generated music
plt.figure(figsize=(10, 4))
S_gen = librosa.feature.melspectrogram(y=y_gen, sr=sr_gen)
librosa.display.specshow(librosa.power_to_db(S_gen, ref=np.max),
                         sr=sr_gen, x_axis='time', y_axis='mel')
plt.title("Generated Music Spectrogram")
plt.colorbar(format="%+2.0f dB")
plt.tight_layout()
plt.show()

## Generating Multiple songs

In [ ]:
def main(num_songs=5):  # You can set how many songs to generate
    print(" Loading dataset and model...")
    import mitdeeplearning as mdl
    songs = mdl.lab1.load_training_data()

    vocab = sorted(set("".join(songs)))
    char2idx = {u: i for i, u in enumerate(vocab)}
    idx2char = np.array(vocab)

    # Define the model
    class LSTMModel(tf.keras.Model):
        def __init__(self, vocab_size, embedding_dim=256, rnn_units=1024):
            super().__init__()
            self.embedding = tf.keras.layers.Embedding(vocab_size, embedding_dim)
            self.lstm = tf.keras.layers.LSTM(rnn_units, return_sequences=True, stateful=True)
            self.dense = tf.keras.layers.Dense(vocab_size)

        def call(self, inputs):
            x = self.embedding(inputs)
            x = self.lstm(x)
            return self.dense(x)

    # Load model
    model = LSTMModel(len(vocab))
    try:
        model.build((1, None))
        model.load_weights("./training_checkpoints/my_ckpt.weights.h5")
        print(" Model loaded successfully")
    except Exception as e:
        print(f" Failed to load model: {e}")
        return

    for i in range(num_songs):
        print(f"\n====================  Pair {i+1} ====================")
        original_song = random.choice(songs)
        seed_string = original_song[:50]

        generated_song = generate_song_from_model(
            model, char2idx, idx2char,
            start_string=seed_string,
            generation_length=1500,
            temperature=0.5
        )

        # Save both songs
        with open(f"original_song_{i+1}.abc", "w") as f:
            f.write(original_song)

        with open(f"generated_song_{i+1}.abc", "w") as f:
            f.write(generated_song)

        # Convert to audio
        original_audio = convert_abc_to_audio(original_song, f"original_song_{i+1}")
        generated_audio = convert_abc_to_audio(generated_song, f"generated_song_{i+1}")

        # Playback
        if original_audio:
            print(f"\n Playing original song {i+1}:")
            display(Audio(filename=original_audio))

        if generated_audio:
            print(f"\n Playing generated song {i+1}:")
            display(Audio(filename=generated_audio))

# Run
if __name__ == "__main__":
    main(num_songs=8)  # Change to desired number